# Vocabulary & Data Preparation

Converts words → numbers (tokens)

Prepares input/output sequences

Adds special tokens (<sos>, <eos>)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# Vocabulary Setup
# -----------------------------
# Define a small vocabulary
vocab = ["<pad>", "<sos>", "<eos>", "i", "like", "ai"]

# Create mappings: word → index and index → word
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# Convert sentence (list of words) → list of indices
def encode(sentence):
    return [word2idx[w] for w in sentence]

# Convert indices → words (for output interpretation)
def decode(indices):
    return [idx2word[i] for i in indices]

# Example input-output pair
input_sentence = ["i", "like", "ai"]
target_sentence = ["ai", "like", "i"]

# Convert to tensors
input_tensor = torch.tensor([encode(input_sentence)])

# Add <sos> and <eos> for decoder
target_tensor = torch.tensor([
    [word2idx["<sos>"]] + encode(target_sentence) + [word2idx["<eos>"]]
])

# Encoder (RNN)

Reads input sequence

Compresses it into a hidden state (context vector)

In [ ]:
# -----------------------------
# Encoder
# -----------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        # Converts token IDs → dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Simple RNN
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        # Step 1: Convert tokens → embeddings -> [batch_size, seq_length, embedding_dimension]
        embedded = self.embedding(x)

        # Step 2: Pass through RNN
        outputs, hidden = self.rnn(embedded)

        # Return only final hidden state (context vector)
        return hidden

# Decoder (RNN)


Takes hidden state from encoder

Generates output one token at a time

In [ ]:
# -----------------------------
# Decoder
# -----------------------------
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)

        # Maps hidden state → vocabulary probabilities
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        # Step 1: Embed input token
        embedded = self.embedding(x)

        # Step 2: Pass through RNN using previous hidden state
        outputs, hidden = self.rnn(embedded, hidden)

        # Step 3: Predict next token
        predictions = self.fc(outputs)

        return predictions, hidden

# Seq2Seq Model (Encoder + Decoder)

Connects encoder and decoder

Implements teacher forcing

In [ ]:
# -----------------------------
# Seq2Seq Model
# -----------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        # Step 1: Encode input sequence
        hidden = self.encoder(src)

        outputs = []

        # First input to decoder is <sos>
        input_token = trg[:, 0].unsqueeze(1)

        # Step 2: Decode step-by-step
        for t in range(1, trg.size(1)):
            output, hidden = self.decoder(input_token, hidden)
            outputs.append(output)

            # Teacher forcing: use actual target token
            input_token = trg[:, t].unsqueeze(1)

        # Concatenate all outputs
        return torch.cat(outputs, dim=1)

# Model Initialization & Training Setup

Defines model size

Sets loss function and optimizer

In [ ]:
# -----------------------------
# Training Setup
# -----------------------------
vocab_size = len(vocab)
embed_size = 8
hidden_size = 16

# Create encoder & decoder
encoder = Encoder(vocab_size, embed_size, hidden_size)
decoder = Decoder(vocab_size, embed_size, hidden_size)

# Combine into Seq2Seq model
model = Seq2Seq(encoder, decoder)

# Loss function (for classification over vocab)
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop

Forward pass

Compute loss

Backpropagation

In [ ]:
# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(300):
    optimizer.zero_grad()

    # Forward pass
    output = model(input_tensor, target_tensor)

    # Reshape outputs for loss calculation
    output = output.view(-1, vocab_size)

    # Target excludes <sos>
    target = target_tensor[:, 1:].reshape(-1)

    # Compute loss
    loss = criterion(output, target)

    # Backpropagation
    loss.backward()
    optimizer.step()

    # Print progress
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.8063
Epoch 50, Loss: 0.0108
Epoch 100, Loss: 0.0046
Epoch 150, Loss: 0.0028
Epoch 200, Loss: 0.0019
Epoch 250, Loss: 0.0014


# Inference (Prediction)

Generates output without teacher forcing

Stops when <eos> is predicted

In [ ]:
# -----------------------------
# Inference
# -----------------------------
def predict(model, src, max_len=10):
    model.eval()

    with torch.no_grad():
        # Encode input
        hidden = model.encoder(src)

        # Start with <sos>
        input_token = torch.tensor([[word2idx["<sos>"]]])

        result = []

        # Generate tokens one by one
        for _ in range(max_len):
            output, hidden = model.decoder(input_token, hidden)

            # Get predicted token
            pred = output.argmax(2).item()

            # Stop at <eos>
            if pred == word2idx["<eos>"]:
                break

            result.append(pred)

            # Feed prediction back as next input
            input_token = torch.tensor([[pred]])

    return decode(result)

# Test prediction
print("Prediction:", predict(model, input_tensor))

Prediction: ['ai', 'like', 'i']
